In [ ]:
"""
Fix Existing JSON Transcripts - Split into Fixed Time Intervals
================================================================
Script ini memproses ulang semua file JSON di folder transcribe_result
dan memotong menjadi interval waktu tetap: 0-30, 30-60, 60-90, dst.

Features:
- Cek database untuk skip video yang sudah di-fix
- Update database setelah fix berhasil
- Backup file original
- Support CJK languages (Chinese, Japanese, Korean) - split per character

Usage:
    python fix_segments.py
"""

In [ ]:
import os
import json
import sqlite3
import re
from glob import glob

In [ ]:
# Configuration - Paths
# In a Jupyter Notebook, __file__ is undefined, and os.getcwd() can vary
# depending on how the notebook is launched. We will find the project root 
# (the directory containing the 'data' folder).
import os

_current_dir = os.getcwd()
# Keep moving up until we find the 'data' folder, or default to current dir if not found
while _current_dir and not os.path.isdir(os.path.join(_current_dir, "data")):
    parent = os.path.dirname(_current_dir)
    if parent == _current_dir:
        break
    _current_dir = parent

PROJECT_ROOT = _current_dir
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

In [ ]:
TRANSCRIBE_FOLDER = os.path.join(DATA_DIR, "transcribe_result")
DATABASE_PATH = os.path.join(DATA_DIR, "youtube.db")

In [ ]:
CHUNK_DURATION = 30  # Fixed chunk duration in seconds (0-30, 30-60, etc.)
BACKUP = True  # Create backup before modifying
SKIP_FIXED = True # Skip videos that already have fixed=1 in database

In [ ]:
# CJK Unicode ranges (Chinese, Japanese, Korean)
CJK_PATTERN = re.compile(
    r"[\u4e00-\u9fff"  # CJK Unified Ideographs (Chinese)
    r"\u3400-\u4dbf"  # CJK Unified Ideographs Extension A
    r"\u3040-\u309f"  # Hiragana (Japanese)
    r"\u30a0-\u30ff"  # Katakana (Japanese)
    r"\uac00-\ud7af"  # Hangul Syllables (Korean)
    r"\u1100-\u11ff]"  # Hangul Jamo (Korean)
)

In [ ]:
def is_cjk_text(text: str) -> bool:
    """Check if text contains significant CJK characters."""
    if not text:
        return False
    cjk_chars = len(CJK_PATTERN.findall(text))
    # If more than 30% of characters are CJK, treat as CJK text
    return cjk_chars / max(len(text), 1) > 0.3

In [ ]:
def smart_split(text: str) -> list:
    """
    Smart split text into tokens.
    - For CJK languages: split per character
    - For other languages: split by whitespace
    - For mixed: split appropriately
    """
    if not text:
        return []

    text = text.strip()

    if is_cjk_text(text):
        # CJK text: split into individual characters (excluding spaces)
        # But keep non-CJK words together
        tokens = []
        current_word = ""

        for char in text:
            if CJK_PATTERN.match(char):
                # CJK character - save any pending word first
                if current_word.strip():
                    tokens.append(current_word.strip())
                    current_word = ""
                # Add CJK char as individual token
                tokens.append(char)
            elif char.isspace():
                # Space - save pending word
                if current_word.strip():
                    tokens.append(current_word.strip())
                    current_word = ""
            else:
                # Non-CJK, non-space - accumulate
                current_word += char

        # Don't forget last word
        if current_word.strip():
            tokens.append(current_word.strip())

        return tokens
    else:
        # Non-CJK text: split by whitespace
        return text.split()

In [ ]:
def smart_join(tokens: list, is_cjk: bool = False) -> str:
    """
    Smart join tokens back into text.
    - For CJK: no spaces between CJK characters
    - For mixed: appropriate spacing
    """
    if not tokens:
        return ""

    if not is_cjk:
        return " ".join(tokens)

    # For CJK, join without spaces between CJK chars
    result = []
    for i, token in enumerate(tokens):
        if i == 0:
            result.append(token)
        else:
            prev_token = tokens[i - 1]
            # Add space only if both are non-CJK
            prev_is_cjk = len(token) == 1 and CJK_PATTERN.match(prev_token[-1:])
            curr_is_cjk = len(token) == 1 and CJK_PATTERN.match(token)

            if prev_is_cjk or curr_is_cjk:
                result.append(token)  # No space
            else:
                result.append(" " + token)  # Add space

    return "".join(result)

In [ ]:
def get_db_connection():
    """Get database connection."""
    return sqlite3.connect(DATABASE_PATH)

In [ ]:
def is_video_fixed(video_id: str) -> bool:
    """Check if video is already fixed in database."""
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT fixed FROM videos WHERE video_id = ?", (video_id,))
        result = cursor.fetchone()
        conn.close()

        if result and result[0] == 1:
            return True
        return False
    except Exception:
        return False

In [ ]:
def mark_video_fixed(video_id: str) -> bool:
    """Mark video as fixed in database."""
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("UPDATE videos SET fixed = 1 WHERE video_id = ?", (video_id,))
        conn.commit()
        affected = cursor.rowcount
        conn.close()
        return affected > 0
    except Exception as e:
        print(f"  ⚠️ Failed to update DB: {e}")
        return False

In [ ]:
def get_unfixed_videos() -> list:
    """Get list of video_ids that are not yet fixed."""
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT video_id FROM videos 
            WHERE text_loc IS NOT NULL 
            AND text_loc != ''
            AND (fixed = 0 OR fixed IS NULL)
        """
        )
        results = [row[0] for row in cursor.fetchall()]
        conn.close()
        return results
    except Exception:
        return []

In [ ]:
def split_into_fixed_intervals(
    segments: list, chunk_duration: float = CHUNK_DURATION
) -> list:
    """
    Split segments into fixed time intervals (0-30, 30-60, 60-90, etc.)

    IMPROVED Algorithm:
    - Distributes words proportionally based on segment duration
    - Uses fractional word distribution to avoid losing words
    - Tracks cumulative word allocation to ensure all words are used

    Args:
        segments: List of segment dicts with 'start', 'end', 'text'
        chunk_duration: Fixed interval duration in seconds (default: 30)

    Returns:
        List of segments with fixed time intervals
    """
    if not segments:
        return []

    result = []
    current_interval_idx = 0
    current_interval_start = 0
    current_interval_end = chunk_duration
    current_interval_words = []

    # Detect if we're dealing with CJK text (check first few segments)
    sample_text = " ".join(seg.get("text", "") for seg in segments[:5])
    is_cjk = is_cjk_text(sample_text)

    for seg in segments:
        seg_start = seg.get("start") or 0
        seg_end = seg.get("end") or seg_start
        seg_text = seg.get("text", "").strip()

        # Skip empty segments
        if not seg_text:
            continue

        # Use smart_split for CJK support
        words = smart_split(seg_text)
        if not words:
            continue

        total_words = len(words)

        # Edge Case 1: Handle zero duration segments - put all words in current interval
        seg_duration = seg_end - seg_start
        if seg_duration <= 0:
            # Advance interval if needed to reach segment start
            while current_interval_end <= seg_start:
                if current_interval_words:
                    result.append(
                        {
                            "start": round(current_interval_start, 2),
                            "end": round(current_interval_end, 2),
                            "text": smart_join(current_interval_words, is_cjk),
                        }
                    )
                    current_interval_words = []
                current_interval_idx += 1
                current_interval_start = current_interval_idx * chunk_duration
                current_interval_end = (current_interval_idx + 1) * chunk_duration

            # Add all words to current interval
            current_interval_words.extend(words)
            continue

        # Edge Case 2: Out of order overlapping segment (segment is entirely in the past)
        if seg_end <= current_interval_start:
            # Drop its words into the current interval instead of infinitely advancing
            current_interval_words.extend(words)
            continue

        # Calculate words per second for this segment
        words_per_sec = total_words / seg_duration

        # Advance interval to segment start if there's a gap
        while current_interval_end <= seg_start:
            if current_interval_words:
                result.append(
                    {
                        "start": round(current_interval_start, 2),
                        "end": round(current_interval_end, 2),
                        "text": smart_join(current_interval_words, is_cjk),
                    }
                )
                current_interval_words = []
            current_interval_idx += 1
            current_interval_start = current_interval_idx * chunk_duration
            current_interval_end = (current_interval_idx + 1) * chunk_duration

        # Track cumulative words allocated (use float for accuracy)
        cumulative_words_allocated = 0.0
        word_idx = 0
        seg_current_time = seg_start

        while word_idx < total_words:
            # Calculate effective start time within segment
            effective_start = max(seg_current_time, current_interval_start)

            # Time remaining in current interval from effective start
            time_in_interval = min(current_interval_end, seg_end) - effective_start

            if time_in_interval <= 0:
                # Need to move to next interval
                if current_interval_words:
                    result.append(
                        {
                            "start": round(current_interval_start, 2),
                            "end": round(current_interval_end, 2),
                            "text": smart_join(current_interval_words, is_cjk),
                        }
                    )
                    current_interval_words = []

                current_interval_idx += 1
                current_interval_start = current_interval_idx * chunk_duration
                current_interval_end = (current_interval_idx + 1) * chunk_duration
                continue

            # Calculate words for this chunk based on time proportion
            # Use cumulative allocation to avoid rounding errors
            time_elapsed_in_seg = min(current_interval_end, seg_end) - seg_start
            target_cumulative_words = (time_elapsed_in_seg / seg_duration) * total_words

            # Words to allocate in this interval (round properly)
            words_to_take = int(
                round(target_cumulative_words - cumulative_words_allocated)
            )

            # Ensure we take at least 1 word if there are remaining words
            if words_to_take <= 0 and word_idx < total_words:
                words_to_take = 1

            # Don't take more than remaining
            words_to_take = min(words_to_take, total_words - word_idx)

            if words_to_take > 0:
                current_interval_words.extend(
                    words[word_idx : word_idx + words_to_take]
                )
                word_idx += words_to_take
                cumulative_words_allocated += words_to_take

            # Update time position
            seg_current_time = min(current_interval_end, seg_end)

            # If we've reached the end of this interval (but not end of segment), save it
            if current_interval_end <= seg_end and word_idx < total_words:
                result.append(
                    {
                        "start": round(current_interval_start, 2),
                        "end": round(current_interval_end, 2),
                        "text": smart_join(current_interval_words, is_cjk),
                    }
                )
                current_interval_words = []

                current_interval_idx += 1
                current_interval_start = current_interval_idx * chunk_duration
                current_interval_end = (current_interval_idx + 1) * chunk_duration
            elif word_idx >= total_words:
                # All words from this segment are allocated
                break
            else:
                # We reached the end of the segment but there are leftover words.
                current_interval_words.extend(words[word_idx:])
                word_idx = total_words
                break

    # Don't forget the last interval
    if current_interval_words:
        # Find actual max end time
        max_end = max(seg.get("end", 0) for seg in segments if seg.get("end"))
        result.append(
            {
                "start": round(current_interval_start, 2),
                "end": round(min(current_interval_end, max_end), 2),
                "text": smart_join(current_interval_words, is_cjk),
            }
        )

    return result

In [ ]:
def process_json_file(filepath: str) -> tuple:
    """
    Process a single JSON file and split long segments.

    Returns:
        Tuple of (original_count, new_count, was_modified)
    """
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        if "segments" not in data:
            print(f"  ⚠️ No 'segments' key found, skipping")
            return 0, 0, False

        original_segments = data["segments"]
        original_count = len(original_segments)

        # Split into fixed time intervals
        new_segments = split_into_fixed_intervals(original_segments, CHUNK_DURATION)
        new_count = len(new_segments)

        # Create backup if enabled (always backup since we're restructuring)
        if BACKUP:
            backup_path = filepath + ".backup"
            if not os.path.exists(backup_path):
                with open(backup_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, indent=2, ensure_ascii=False)

        # Save modified file
        data["segments"] = new_segments
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        return original_count, new_count, True

    except json.JSONDecodeError as e:
        print(f"  ❌ JSON parse error: {e}")
        return 0, 0, False
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return 0, 0, False

In [ ]:
def main():
    print("=" * 70)
    print("FIX SEGMENTS - Split into Fixed Time Intervals")
    print("=" * 70)
    print(f"📂 Folder: {TRANSCRIBE_FOLDER}")
    print(f"💾 Database: {DATABASE_PATH}")
    print(f"⏱️ Interval: {CHUNK_DURATION} seconds (0-30, 30-60, 60-90, ...)")
    print(f"💾 Backup: {'Yes' if BACKUP else 'No'}")
    print(f"⏭️ Skip fixed: {'Yes' if SKIP_FIXED else 'No'}")
    print("=" * 70)

    # Find all JSON files (excluding backup files)
    json_pattern = os.path.join(TRANSCRIBE_FOLDER, "*.json")
    all_files = glob(json_pattern)
    json_files = [f for f in all_files if not f.endswith(".backup")]

    if not json_files:
        print(f"❌ No JSON files found in {TRANSCRIBE_FOLDER}")
        return

    # Get list of unfixed videos from DB if SKIP_FIXED is enabled
    unfixed_video_ids = get_unfixed_videos() if SKIP_FIXED else None

    print(f"📋 Found {len(json_files)} JSON files")
    if SKIP_FIXED and unfixed_video_ids:
        print(f"⏳ Unfixed videos in DB: {len(unfixed_video_ids)}")
    print()

    total_original = 0
    total_new = 0
    modified_count = 0
    skipped_count = 0

    for filepath in json_files:
        filename = os.path.basename(filepath)
        video_id = os.path.splitext(filename)[0]  # Remove .json extension

        # Skip if already fixed in database
        if SKIP_FIXED and is_video_fixed(video_id):
            print(f"⏭️ Skipping (already fixed): {filename}")
            skipped_count += 1
            continue

        print(f"📄 Processing: {filename}")

        original, new, was_modified = process_json_file(filepath)

        if was_modified:
            print(f"   ✂️ Split: {original} → {new} segments")
            modified_count += 1

            # Mark as fixed in database
            if mark_video_fixed(video_id):
                print(f"   ✅ Marked as fixed in database")
            else:
                print(f"   ⚠️ Video not found in database (standalone file)")
        else:
            print(f"   ⚠️ No changes ({original} segments)")

        total_original += original
        total_new += new

    print("\n" + "=" * 70)
    print("📊 Summary:")
    print(f"   📁 Files found: {len(json_files)}")
    print(f"   ⏭️ Skipped (already fixed): {skipped_count}")
    print(f"   ✂️ Files modified: {modified_count}")
    print(f"   📝 Total segments: {total_original} → {total_new}")
    if BACKUP and modified_count > 0:
        print(f"   💾 Backups created: {modified_count} (.backup files)")
    print("=" * 70)

In [ ]:
if __name__ == "__main__":
    main()